In [20]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
import plotly.express as px

In [10]:
train_traffic = pd.read_csv('../data/df_train_traffic_encoded_scaled.csv')
train_average_bill = pd.read_csv('../data/df_train_average_bill_encoded_scaled.csv')

X_train_traffic = train_traffic.drop(columns=['Трафик'])
y_train_traffic = train_traffic['Трафик']

X_train_average_bill = train_average_bill.drop(columns=['Средний чек'])
y_train_average_bill = train_average_bill['Средний чек']

X_train_traffic, X_val_traffic, y_train_traffic, y_val_traffic = train_test_split(X_train_traffic, y_train_traffic, test_size=0.3, random_state=598)
X_train_train_traffic, X_train_test_traffic, y_train_train_traffic, y_train_test_traffic = train_test_split(X_train_traffic, y_train_traffic, test_size=0.28, random_state=598)

X_train_average_bill, X_val_average_bill, y_train_average_bill, y_val_average_bill = train_test_split(X_train_average_bill, y_train_average_bill, test_size=0.3, random_state=598)
X_train_train_average_bill, X_train_test_average_bill, y_train_train_average_bill, y_train_test_average_bill = train_test_split(X_train_average_bill, y_train_average_bill, test_size=0.28, random_state=598)

In [11]:
def metrics_compare(model, model_name, target):
    
    model_rmse = 0
    model_mae = 0

    if target == 'Трафик':

        model.fit(X_train_train_traffic, y_train_train_traffic)

        model_rmse = np.sqrt(mean_squared_error(y_train_test_traffic, model.predict(X_train_test_traffic)))
        model_mae = mean_absolute_error(y_train_test_traffic, model.predict(X_train_test_traffic))

    elif target == 'Средний чек':

        model.fit(X_train_train_average_bill, y_train_train_average_bill)

        model_rmse = np.sqrt(mean_squared_error(y_train_test_average_bill, model.predict(X_train_test_average_bill)))
        model_mae = mean_absolute_error(y_train_test_average_bill, model.predict(X_train_test_average_bill))

    model_metrics = pd.DataFrame({
        'Целевая переменная' : [target],
        'Модель' : [model_name],
        'RMSE' : [model_rmse],
        'MAE' : [model_mae]
    })

    return model_metrics

## Трафик

In [12]:
rf_best_params_traffic = {
    'n_estimators': [1000],
    'max_depth': [60],
    'max_features': [0.8],
    'max_samples': [1.0]
}

In [13]:
mlp_metrics_traffic = pd.DataFrame({
    'Целевая переменная' : ['Трафик'],
    'Модель' : ['MLP'],
    'RMSE' : [5605.07],
    'MAE' : [4021.38]
})
mlp_metrics_traffic

,Целевая переменная,Модель,RMSE,MAE
0,Трафик,MLP,5605.07,4021.38


In [14]:
ridge_metrics_traffic = metrics_compare(
    Ridge(alpha=1.0),
    'RidgeRegression',
    'Трафик'
)

knn_metrics_traffic = metrics_compare(
    KNeighborsRegressor(n_neighbors=1, n_jobs=-1),
    'KNN',
    'Трафик'
)

rf_metrics_traffic = metrics_compare(
    RandomForestRegressor(
        max_depth=60,
        max_features=0.8,
        max_samples=1.0,
        n_estimators=1000,
        random_state=598
    ),
    'RandomForest',
    'Трафик'
)

catboost_metrics_traffic = metrics_compare(
    CatBoostRegressor(learning_rate=0.05, l2_leaf_reg=1, iterations=500, depth=10, random_state=1337),
    'CATBOOST',
    'Трафик'
)

0:	learn: 13159.3617566	total: 35.8ms	remaining: 17.8s
1:	learn: 12898.6730812	total: 49.4ms	remaining: 12.3s
2:	learn: 12660.9376491	total: 59.3ms	remaining: 9.83s
3:	learn: 12438.6187988	total: 76.3ms	remaining: 9.47s
4:	learn: 12232.7403967	total: 85.1ms	remaining: 8.43s
5:	learn: 12044.0050064	total: 93.9ms	remaining: 7.73s
6:	learn: 11868.6926380	total: 102ms	remaining: 7.21s
7:	learn: 11706.9420393	total: 112ms	remaining: 6.86s
8:	learn: 11554.8218567	total: 121ms	remaining: 6.58s
9:	learn: 11412.9152828	total: 129ms	remaining: 6.34s
10:	learn: 11279.9147564	total: 138ms	remaining: 6.13s
11:	learn: 11158.5499265	total: 146ms	remaining: 5.95s
12:	learn: 11047.8821321	total: 154ms	remaining: 5.78s
13:	learn: 10942.9744644	total: 163ms	remaining: 5.64s
14:	learn: 10847.7058259	total: 171ms	remaining: 5.53s
15:	learn: 10758.8438513	total: 180ms	remaining: 5.44s
16:	learn: 10675.1485484	total: 189ms	remaining: 5.36s
17:	learn: 10598.2701384	total: 198ms	remaining: 5.29s
18:	learn: 105

In [15]:
metrics_traffic = pd.concat([ridge_metrics_traffic, knn_metrics_traffic, rf_metrics_traffic, catboost_metrics_traffic, mlp_metrics_traffic], axis=0)
metrics_traffic

,Целевая переменная,Модель,RMSE,MAE
0,Трафик,RidgeRegression,10093.962408,7647.526369
0,Трафик,KNN,6785.831310,3856.275327
0,Трафик,RandomForest,3843.241801,2280.605389
0,Трафик,CATBOOST,7275.962785,5424.699857
0,Трафик,MLP,5605.070000,4021.380000


In [25]:
fig = px.bar(
    metrics_traffic.sort_values(['RMSE', 'MAE']),
    x='Модель',
    y=['RMSE', 'MAE'],
    barmode='group',
    title='Результаты функционала ошибок для предсказания трафика'
)

fig.show()

## Средний чек

In [16]:
rf_best_params_average_bill = {
    'n_estimators': [1000],
    'max_depth': [50],
    'max_features': [0.63],
    'max_samples': [1.0]
}

In [17]:
mlp_metrics_average_bill = pd.DataFrame({
    'Целевая переменная' : ['Средний чек'],
    'Модель' : ['MLP'],
    'RMSE' : [121.62],
    'MAE' : [89.13]
})
mlp_metrics_average_bill

,Целевая переменная,Модель,RMSE,MAE
0,Средний чек,MLP,121.62,89.13


In [18]:
ridge_metrics_average_bill = metrics_compare(
    Ridge(alpha=1.0),
    'RidgeRegression',
    'Средний чек'
)

knn_metrics_average_bill = metrics_compare(
    KNeighborsRegressor(n_neighbors=1, n_jobs=-1),
    'KNN',
    'Средний чек'
)

rf_metrics_average_bill = metrics_compare(
    RandomForestRegressor(
        max_depth=50,
        max_features=0.63,
        max_samples=1.0,
        n_estimators=1000,
        random_state=598
    ),
    'RandomForest',
    'Средний чек'
)

catboost_metrics_average_bill = metrics_compare(
    CatBoostRegressor(learning_rate=0.05, l2_leaf_reg=1, iterations=500, depth=10, random_state=1337),
    'CATBOOST',
    'Средний чек'
)

0:	learn: 305.6671470	total: 24.7ms	remaining: 12.3s
1:	learn: 297.9752173	total: 39.9ms	remaining: 9.93s
2:	learn: 290.7986506	total: 48.8ms	remaining: 8.09s
3:	learn: 284.0972895	total: 57.4ms	remaining: 7.12s
4:	learn: 277.9422236	total: 66.1ms	remaining: 6.54s
5:	learn: 272.1493150	total: 74.7ms	remaining: 6.15s
6:	learn: 266.8633939	total: 83.1ms	remaining: 5.85s
7:	learn: 261.8478661	total: 91.3ms	remaining: 5.62s
8:	learn: 257.2179963	total: 100ms	remaining: 5.45s
9:	learn: 252.9871890	total: 108ms	remaining: 5.3s
10:	learn: 249.0682215	total: 118ms	remaining: 5.25s
11:	learn: 245.4204684	total: 126ms	remaining: 5.14s
12:	learn: 242.0732158	total: 135ms	remaining: 5.04s
13:	learn: 238.9519388	total: 143ms	remaining: 4.97s
14:	learn: 236.1056771	total: 153ms	remaining: 4.94s
15:	learn: 233.4759463	total: 161ms	remaining: 4.88s
16:	learn: 231.0035490	total: 170ms	remaining: 4.83s
17:	learn: 228.7500926	total: 179ms	remaining: 4.79s
18:	learn: 226.6148657	total: 187ms	remaining: 4.

In [19]:
metrics_average_bill = pd.concat([ridge_metrics_average_bill, knn_metrics_average_bill, rf_metrics_average_bill, catboost_metrics_average_bill, mlp_metrics_average_bill], axis=0)
metrics_average_bill

,Целевая переменная,Модель,RMSE,MAE
0,Средний чек,RidgeRegression,215.383689,161.289115
0,Средний чек,KNN,160.983053,99.129364
0,Средний чек,RandomForest,93.727114,54.420618
0,Средний чек,CATBOOST,147.282291,110.115302
0,Средний чек,MLP,121.620000,89.130000


In [26]:
fig = px.bar(
    metrics_average_bill.sort_values(['RMSE', 'MAE']),
    x='Модель',
    y=['RMSE', 'MAE'],
    barmode='group',
    title='Результаты функционала ошибок для предсказания среднего чека'
)

fig.show()